In [11]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from torch.optim.lr_scheduler import ReduceLROnPlateau
from optional_train import DigitNet, DfToDataset, train_epoch, eval_model

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используется: {device}")

Используется: cuda


In [13]:
config = {
    'channels': [32, 64],
    'kernel_size': 3,
    'dropout': 0.3,
    'mlp_dim': 128,
    'pool': 2,
    'stride': 2
}

EPOCHS = 3

In [14]:
X_df = pd.read_csv('data/train.csv')
y = X_df['label']

In [15]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_metrics = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_df, y)):
    X_df_train = X_df.iloc[train_idx]
    X_df_val = X_df.iloc[val_idx]

    X_train = DfToDataset(X_df_train, is_test=False)
    X_val = DfToDataset(X_df_val, is_test=False)

    train_loader = DataLoader(X_train, batch_size=64, shuffle=True)
    val_loader = DataLoader(X_val, batch_size=64, shuffle=False)

    model = DigitNet(config).to(device)
    optimizer = AdamW(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss().to(device)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)

    best_fold_metric = 0

    best_fold_metric = 0
    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, optimizer, train_loader, loss_fn, scheduler, device)
        val_metric = eval_model(model, val_loader, device)

        scheduler.step(val_metric)

        print(f"Эпоха {epoch + 1}/{EPOCHS} | Loss: {train_loss:.4f} | Val Acc: {val_metric:.4f}")

        if val_metric > best_fold_metric:
            best_fold_metric = val_metric

    fold_metrics.append(best_fold_metric)
    print(f"Лучший результат фолда {fold + 1}: {best_fold_metric}")

    torch.save(model.state_dict(), f'models/model_fold_{fold + 1}.pt')
    print(f"Модель фолда {fold + 1} сохранена на диск")


print(f"Средний результат по фолдам: {np.mean(fold_metrics)}")


Эпоха 1/3 | Loss: 0.2915 | Val Acc: 0.9739
Эпоха 2/3 | Loss: 0.0855 | Val Acc: 0.9829
Эпоха 3/3 | Loss: 0.0629 | Val Acc: 0.9830
Лучший результат фолда 1: 0.9829761904761904
Модель фолда 1 сохранена на диск
Эпоха 1/3 | Loss: 0.3200 | Val Acc: 0.9727
Эпоха 2/3 | Loss: 0.0964 | Val Acc: 0.9785
Эпоха 3/3 | Loss: 0.0665 | Val Acc: 0.9825
Лучший результат фолда 2: 0.9825
Модель фолда 2 сохранена на диск
Эпоха 1/3 | Loss: 0.2857 | Val Acc: 0.9761
Эпоха 2/3 | Loss: 0.0846 | Val Acc: 0.9832
Эпоха 3/3 | Loss: 0.0603 | Val Acc: 0.9846
Лучший результат фолда 3: 0.9846428571428572
Модель фолда 3 сохранена на диск
Эпоха 1/3 | Loss: 0.3199 | Val Acc: 0.9780
Эпоха 2/3 | Loss: 0.0902 | Val Acc: 0.9839
Эпоха 3/3 | Loss: 0.0670 | Val Acc: 0.9867
Лучший результат фолда 4: 0.9866666666666667
Модель фолда 4 сохранена на диск
Эпоха 1/3 | Loss: 0.3000 | Val Acc: 0.9763
Эпоха 2/3 | Loss: 0.0895 | Val Acc: 0.9824
Эпоха 3/3 | Loss: 0.0641 | Val Acc: 0.9854
Лучший результат фолда 5: 0.9853571428571428
Модель фол